# RDD reconstruction example

In [1]:
from sfm import match_rdd, extract_rdd
from hloc import (
    extract_features,
    reconstruction,
    visualization,
    pairs_from_retrieval,
    pairs_from_exhaustive,
)
from pathlib import Path
import os
import torch

/wekafs/ict/gonglinc/Documents/feature-matching/rdd_new/src/matchers/lightglue.py:29: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/home/gonglinc/Documents/miniconda3/envs/rdd/lib/python3.10/site-packages/torch/functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at ../aten/src/ATen/native/TensorShape.cpp:3595.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/wekafs/ict/gonglinc/Documents/feature-matching/rdd_new/third_party/alike/alike.py:48: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorc

In [ ]:
images_dir = Path('./assets/images')
device = torch.cuda.is_available()
images = [image for image in os.listdir(images_dir) if image.endswith('.jpg') or image.endswith('.png')]
outputs = Path('./outputs/reconstruction')
if not outputs.exists():
    outputs.mkdir(parents=True)
sfm_pairs = outputs / 'sfm_pairs.txt'
retrieval_conf = extract_features.confs["netvlad"]
feature_conf = extract_rdd.confs["rdd"]
matcher_conf = match_rdd.confs["rdd+lightglue"]
feature_conf['model']['max_keypoints'] = 4096
exhaustive_if_less = 30
num_matched = 20

In [3]:
# image_retrieval
if len(images) < exhaustive_if_less:
    pairs_from_exhaustive.main(sfm_pairs, images)
else:
    retrieval_path = extract_features.main(retrieval_conf, images_dir, outputs)
    pairs_from_retrieval.main(retrieval_path, sfm_pairs, num_matched=num_matched)
    
# feature_extraction
feature_path = extract_rdd.main(feature_conf, images_dir, outputs)
# matching
match_path = match_rdd.main(matcher_conf, sfm_pairs, feature_conf['output'], outputs)

[2025/09/30 11:36:49 hloc INFO] Found 45 pairs.
[2025/09/30 11:36:49 hloc INFO] Extracting local features with configuration:
{'model': {'config_path': './configs/default.yaml',
           'max_keypoints': 8192,
           'weights': './weights/RDDNext.pth'},
 'output': 'feats-rdd-n4096',
 'preprocessing': {'grayscale': False,
                   'resize_force': True,
                   'resize_max': 1024}}
[2025/09/30 11:36:49 hloc INFO] Found 10 images in root assets/images.
[2025/09/30 11:36:49 hloc INFO] Skipping the extraction.
[2025/09/30 11:36:49 hloc INFO] Matching local features with configuration:
{'model': {'features': 'rdd', 'name': 'lightglue'},
 'output': 'matches-rdd-lightglue'}
[2025/09/30 11:36:49 hloc INFO] Skipping the matching.


Using device: cuda


In [4]:
# reconstruction
image_options = {}
mapper_options = {}
model = reconstruction.main(outputs, images_dir, sfm_pairs, feature_path, 
            match_path, verbose=True, camera_mode='PER_IMAGE', image_options=image_options, mapper_options=mapper_options,
            min_match_score = 0.2, skip_geometric_verification=False)

[2025/09/30 11:36:49 hloc WARNING] The database already exists, deleting it.
[2025/09/30 11:36:49 hloc INFO] Creating an empty database...
[2025/09/30 11:36:49 hloc INFO] Importing images into the database...
[2025/09/30 11:36:50 hloc INFO] Importing features into the database...
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:00<00:00, 1085.01it/s]
[2025/09/30 11:36:50 hloc INFO] Importing matches into the database...
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 45/45 [00:00<00:00, 1315.51it/s]
[2025/09/30 11:36:50 hlo

In [5]:
print(model.summary())

Reconstruction:
	num_rigs = 5
	num_cameras = 5
	num_frames = 5
	num_reg_frames = 5
	num_images = 5
	num_points3D = 429
	num_observations = 1402
	mean_track_length = 3.26807
	mean_observations_per_image = 280.4
	mean_reprojection_error = 0.858315
